# Gold: incremental SCD load

Loads the gold tables from silver (notebook 03 output), after notebook 04 has created them.

- **dim_customer, dim_address, dim_product**: SCD2. A changed attribute closes the current version and opens a new one.
- **dim_category**: SCD1. Overwritten in place.
- **dim_date**: loaded once by notebook 04 (SCD0).
- **bridge_product_category, fact_sales, fact_payment, fact_return**: upserted by business key. Dimension keys are point-in-time, so a fact keeps the dimension version that was valid when it happened.

Each load reads its watermark from `audit.gold_load_audit`, so only rows changed since the last successful load are read. Rerunning the notebook writes nothing new.

In [ ]:
import sys
from dataclasses import asdict
from pathlib import Path

# Make src/ importable when the notebook runs from anywhere inside the project
root = Path.cwd()
while not (root / 'src').exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / 'src'))
sys.path.insert(0, str(root))   # config.py lives at the project root

from pipeline.audit import AuditRun, combine_sinks, make_delta_sink
from pipeline.catalog import ensure_schema_exists, get_table_fqn, assert_table_exists
from pipeline.errors import step
from pipeline.reader import read_table
from pipeline.session import get_spark_session

from gold.dimensions import (
    ADDRESS, CUSTOMER, PRODUCT, shape_address, shape_category, shape_customer, shape_product,
)
from gold.facts import (
    bridge_source, fact_payment_source, fact_return_source, fact_sales_source,
    shape_bridge, shape_fact_payment, shape_fact_return, shape_fact_sales,
)
from gold.incremental import incremental_rows, increment_with_repairs, last_watermark, next_watermark, with_change_timestamps
from gold.lookup import count_unknown
from gold.merge import merge_upsert
from gold.scd.scd1 import merge_scd1
from gold.scd.scd2 import merge_scd2, validate_scd2

from config import get_email_mask_salt

spark = get_spark_session('gold_scd_load')


In [ ]:
import logging

from pipeline.events import EventBus
from pipeline.log import make_log_subscriber

logging.basicConfig(format='%(asctime)s %(levelname)s %(message)s', level=logging.INFO)

bus = EventBus()
bus.subscribe(make_log_subscriber(logging.getLogger('pipeline')))


In [ ]:
# Silver is written by notebook 03 to spark_catalog; gold by notebook 04 to forge
SILVER_CATALOG, GOLD_CATALOG = 'spark_catalog', 'forge'
SILVER_SCHEMA, GOLD_SCHEMA, AUDIT_SCHEMA = 'silver', 'gold', 'audit'

SILVER = {e: get_table_fqn(SILVER_CATALOG, SILVER_SCHEMA, e) for e in (
    'customer', 'address', 'product', 'category', 'product_category', 'order', 'order_item',
    'payment', 'return', 'return_item',
)}
GOLD = {g: get_table_fqn(GOLD_CATALOG, GOLD_SCHEMA, g) for g in (
    'dim_customer', 'dim_address', 'dim_product', 'dim_category', 'dim_date',
    'bridge_product_category', 'fact_sales', 'fact_payment', 'fact_return',
)}
AUDIT_TABLE = get_table_fqn(GOLD_CATALOG, AUDIT_SCHEMA, 'gold_load_audit')

EMAIL_SALT = get_email_mask_salt()   # raises unless EMAIL_MASK_SALT is set: customer emails are hashed, never stored

for schema in (GOLD_SCHEMA, AUDIT_SCHEMA):
    ensure_schema_exists(spark, GOLD_CATALOG, schema)
for table_fqn in [*SILVER.values(), *GOLD.values()]:
    assert_table_exists(spark, table_fqn)   # gold comes from notebook 04, silver from notebook 03


In [ ]:
records = []  # the audit sink for this session: every AuditRun appends here and to the audit table
sink = combine_sinks(records.append, make_delta_sink(spark, AUDIT_TABLE))


def _text(value) -> str:
    return value.isoformat(sep=' ') if value else 'none'


def _record(run, result) -> None:
    for name, value in asdict(result).items():
        run.add(name, value)


def read_silver(entity: str):
    return with_change_timestamps(read_table(spark, SILVER[entity]))


def with_repairs(source, watermark, target: str, business_key: str, key_columns):
    """The increment, plus rows whose earlier load used the unknown member (see increment_with_repairs)."""
    return increment_with_repairs(source, watermark, spark.read.table(target), business_key, key_columns)


def attribute_columns(target: str, keys) -> list[str]:
    """Columns compared to detect changes: everything except keys and pipeline metadata."""
    return [c for c in spark.read.table(target).columns if c not in keys and not c.startswith('_')]


def run_load(name: str, target: str, business_key: str, select, write):
    """One audited load. select(watermark) returns the rows to process; write(run, rows) applies them."""
    with AuditRun('gold_load', name, target, sink, bus=bus) as run:
        watermark = last_watermark(spark, AUDIT_TABLE, target, business_key)
        run.add('watermark_from', _text(watermark))
        with step('select_increment', bus=bus, target=name):
            increment = select(watermark)
            run.add('increment_row_count', increment.count())
        with step('write', bus=bus, target=name):
            write(run, increment)
        with step('record_watermark', bus=bus, target=name):
            run.add('high_watermark', _text(next_watermark(watermark, increment)))
    return records[-1]


In [ ]:
# Dimensions: SCD2 keeps history (customer, address, product)
def scd2_load(table, entity: str, shape):
    target = GOLD[table.name]

    def select(watermark):
        return incremental_rows(read_silver(entity), watermark)

    def write(run, increment):
        _record(run, merge_scd2(spark, target, table, shape(increment), run_id=run.record.run_id))

    return run_load(table.name, target, table.business_key, select, write)


# dim_category: SCD1 overwrites in place (no history)
def category_load():
    target = GOLD['dim_category']

    def select(watermark):
        return incremental_rows(read_silver('category'), watermark)

    def write(run, increment):
        shaped = shape_category(increment, run.record.run_id)
        _record(run, merge_scd1(spark, target, shaped, business_key='category_id'))

    return run_load('dim_category', target, 'category_id', select, write)


# bridge_product_category: links to the product version and category valid when the link started
def bridge_load():
    target = GOLD['bridge_product_category']

    def select(watermark):
        return incremental_rows(bridge_source(read_silver('product_category')), watermark)

    def write(run, increment):
        shaped = shape_bridge(
            increment,
            spark.read.table(GOLD['dim_product']),
            spark.read.table(GOLD['dim_category']),
            run.record.run_id,
        ).cache()
        keys = ('bridge_sk', 'product_category_id')
        _record(run, merge_upsert(spark, target, shaped, business_key='product_category_id',
                                  changed_columns=attribute_columns(target, keys)))
        shaped.unpersist()

    return run_load('bridge_product_category', target, 'product_category_id', select, write)


In [ ]:
# Facts: point-in-time keys. Each fact gets the dimension versions that were valid at the event time.
SALES_FK = ('customer_sk', 'billing_address_sk', 'shipping_address_sk', 'product_sk', 'order_date_sk')
PAYMENT_FK = ('customer_sk', 'payment_date_sk')
RETURN_FK = ('customer_sk', 'product_sk', 'return_date_sk')


def sales_load():
    target = GOLD['fact_sales']
    source = fact_sales_source(read_silver('order_item'), read_silver('order'))

    def select(watermark):
        return with_repairs(source, watermark, target, 'order_item_id', SALES_FK)

    def write(run, increment):
        shaped = shape_fact_sales(
            increment,
            spark.read.table(GOLD['dim_customer']),
            spark.read.table(GOLD['dim_address']),
            spark.read.table(GOLD['dim_product']),
            spark.read.table(GOLD['dim_date']),
            run.record.run_id,
        ).cache()
        for name, value in count_unknown(shaped, SALES_FK).items():
            run.add(name, value)
        keys = ('sales_sk', 'order_item_id')
        _record(run, merge_upsert(spark, target, shaped, business_key='order_item_id',
                                  changed_columns=attribute_columns(target, keys)))
        shaped.unpersist()

    return run_load('fact_sales', target, 'order_item_id', select, write)


def payment_load():
    target = GOLD['fact_payment']
    source = fact_payment_source(read_silver('payment'))

    def select(watermark):
        return with_repairs(source, watermark, target, 'payment_id', PAYMENT_FK)

    def write(run, increment):
        shaped = shape_fact_payment(
            increment,
            spark.read.table(GOLD['dim_customer']),
            spark.read.table(GOLD['dim_date']),
            run.record.run_id,
        ).cache()
        for name, value in count_unknown(shaped, PAYMENT_FK).items():
            run.add(name, value)
        keys = ('payment_sk', 'payment_id')
        _record(run, merge_upsert(spark, target, shaped, business_key='payment_id',
                                  changed_columns=attribute_columns(target, keys)))
        shaped.unpersist()

    return run_load('fact_payment', target, 'payment_id', select, write)


def return_load():
    target = GOLD['fact_return']
    source = fact_return_source(read_silver('return_item'), read_silver('return'))

    def select(watermark):
        return with_repairs(source, watermark, target, 'return_item_id', RETURN_FK)

    def write(run, increment):
        shaped = shape_fact_return(
            increment,
            spark.read.table(GOLD['dim_customer']),
            spark.read.table(GOLD['dim_product']),
            spark.read.table(GOLD['dim_date']),
            run.record.run_id,
        ).cache()
        for name, value in count_unknown(shaped, RETURN_FK).items():
            run.add(name, value)
        keys = ('return_sk', 'return_item_id')
        _record(run, merge_upsert(spark, target, shaped, business_key='return_item_id',
                                  changed_columns=attribute_columns(target, keys)))
        shaped.unpersist()

    return run_load('fact_return', target, 'return_item_id', select, write)


In [ ]:
def run_all_loads():
    """Dimensions first, then the bridge (it needs them), then facts (they need every dimension)."""
    results = [
        scd2_load(CUSTOMER, 'customer', lambda df: shape_customer(df, EMAIL_SALT)),
        scd2_load(ADDRESS, 'address', shape_address),
        scd2_load(PRODUCT, 'product', shape_product),
        category_load(),
        bridge_load(),
        sales_load(),
        payment_load(),
        return_load(),
    ]
    for record in results:
        print(f"{record.entity:24} {record.status:8}", {k: v for k, v in record.metrics.items() if k != 'stage_table'})
    return results


first_load = run_all_loads()


In [ ]:
# Rerun with no source change: the watermark skips the old rows, so nothing new should be written
rerun = run_all_loads()
for record in rerun:
    written = {k: v for k, v in record.metrics.items() if k in ('versions_inserted', 'versions_closed', 'inserted', 'updated')}
    print(f"{record.entity:24} {record.status:8} written={written}")


In [ ]:
# Integrity: one current version per key, and versions join end to end
for table in (CUSTOMER, ADDRESS, PRODUCT):
    print(table.name, validate_scd2(spark, GOLD[table.name], table))

for name in ('fact_sales', 'fact_payment', 'fact_return'):
    fk_columns = {'fact_sales': SALES_FK, 'fact_payment': PAYMENT_FK, 'fact_return': RETURN_FK}[name]
    print(name, count_unknown(spark.read.table(GOLD[name]), fk_columns))

for name, fqn in GOLD.items():
    print(f'{name:26} {spark.read.table(fqn).count()} rows')


In [ ]:
# Audit log: one row per load, with the watermark and counts in metrics
from IPython.display import display

display(
    spark.read.table(AUDIT_TABLE)
    .where("stage = 'gold_load'")
    .select('entity', 'status', 'started_at', 'error_message')
    .orderBy('started_at')
    .toPandas()
)
